In [1]:
import pandas as pd
import os

In [27]:
"""
Final Columns for Data Load
TABLE fact_gen(
  date_id INT,
  state_id INT,
  fuel_id INT,
  generation DECIMAL(12, 2)

TABLE fact_prices(
  date_id INT,
  state_id INT,
  sector_id INT,
  price_per_kwh DECIMAL(10,4),

TABLE fact_fuel_prices(
  date_id INT,
  state_id INT,
  fuel_id INT,
  price_per_mmbtu DECIMAL(10,4),

"""

'\nFinal Columns for Data Load\nTABLE fact_gen(\n  date_id INT,\n  state_id INT,\n  fuel_id INT,\n  generation DECIMAL(12, 2)\n\nTABLE fact_prices(\n  date_id INT,\n  state_id INT,\n  sector_id INT,\n  price_per_kwh DECIMAL(10,4),\n\nTABLE fact_fuel_prices(\n  date_id INT,\n  state_id INT,\n  fuel_id INT,\n  price_per_mmbtu DECIMAL(10,4),\n\n'

In [2]:
gen_df = pd.read_csv("../data/gen_data.csv")
gen_df.head()

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2025-03,90,Pacific,1,Electric Utility,ALL,all fuels,17842.73950,thousand megawatthours
1,2025-03,90,Pacific,1,Electric Utility,AOR,all renewables,822.00528,thousand megawatthours
2,2025-03,90,Pacific,1,Electric Utility,BIO,biomass,37.41079,thousand megawatthours
3,2025-03,90,Pacific,1,Electric Utility,COL,"coal, excluding waste coal",36.42772,thousand megawatthours
4,2025-03,90,Pacific,1,Electric Utility,COW,all coal products,36.42772,thousand megawatthours


In [3]:
price_df = pd.read_csv("../data/price_data.csv")
price_df.head()

,period,stateid,stateDescription,sectorid,sectorName,price,price-units
0,2025-03,AK,Alaska,ALL,all sectors,22.96,cents per kilowatt-hour
1,2025-03,AK,Alaska,COM,commercial,22.10,cents per kilowatt-hour
2,2025-03,AK,Alaska,IND,industrial,20.17,cents per kilowatt-hour
3,2025-03,AK,Alaska,OTH,other,NaN,cents per kilowatt-hour
4,2025-03,AK,Alaska,RES,residential,25.79,cents per kilowatt-hour


In [4]:
fuel_df = pd.read_csv("../data/fuel_data.csv")
fuel_df.head()

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
0,2025-03,SAR,USA-AR,EPG0,Natural Gas,PEU,Electric Power Price,N3045AR3,Arkansas Natural Gas Price Sold to Electric Po...,NaN,$/MCF
1,2025-03,SNE,USA-NE,EPG0,Natural Gas,PEU,Electric Power Price,N3045NE3,Nebraska Natural Gas Price Sold to Electric Po...,7.28,$/MCF
2,2025-03,SMI,USA-MI,EPG0,Natural Gas,PEU,Electric Power Price,N3045MI3,Michigan Natural Gas Price Sold to Electric Po...,NaN,$/MCF
3,2025-03,SCO,COLORADO,EPG0,Natural Gas,PEU,Electric Power Price,N3045CO3,Colorado Natural Gas Price Sold to Electric Po...,NaN,$/MCF
4,2025-03,SAK,USA-AK,EPG0,Natural Gas,PEU,Electric Power Price,N3045AK3,Alaska Natural Gas Price Sold to Electric Powe...,8.34,$/MCF


In [18]:
# Dealing with states
# the to_dict part drops all the duplicates
map = pd.Series(price_df.stateDescription.values, index=price_df.stateid).to_dict()

In [19]:
len(map)

62

In [17]:
state_series = pd.Series(price_df.stateDescription.values, index=price_df.stateid).drop_duplicates()
state_series

stateid
AK                 Alaska
AL                Alabama
AR               Arkansas
AZ                Arizona
CA             California
              ...        
WI              Wisconsin
WNC    West North Central
WSC    West South Central
WV          West Virginia
WY                Wyoming
Length: 62, dtype: str

In [25]:
state_df = state_series.reset_index()

In [24]:
state_df

,stateid,0
0,AK,Alaska
1,AL,Alabama
2,AR,Arkansas
3,AZ,Arizona
4,CA,California
...,...,...
57,WI,Wisconsin
58,WNC,West North Central
59,WSC,West South Central
60,WV,West Virginia


In [28]:
state_df.columns

Index(['state_short', 0], dtype='object')

In [29]:
state_df = state_df.rename(columns = {"stateid": "state_short", 0: "state_long"})
state_df

,state_short,state_long
0,AK,Alaska
1,AL,Alabama
2,AR,Arkansas
3,AZ,Arizona
4,CA,California
...,...,...
57,WI,Wisconsin
58,WNC,West North Central
59,WSC,West South Central
60,WV,West Virginia


## Dealing with Fuel Types
- gen_df and fuel df need to share the same keymap for the fuel id

In [ ]:
gen_df["fuelTypeDescription"].unique()

In [ ]:
gen_df["fueltypeid"].unique()

In [17]:
df=gen_df.drop_duplicates(subset = ["fueltypeid", "fuelTypeDescription"])

In [19]:
df[["fueltypeid", "fuelTypeDescription"]]

,fueltypeid,fuelTypeDescription
0,ALL,all fuels
1,AOR,all renewables
2,BIO,biomass
3,COL,"coal, excluding waste coal"
4,COW,all coal products
5,DFO,distillate fuel oil
6,FOS,fossil fuels
7,GEO,geothermal
8,HPS,hydro-electric pumped storage
9,HYC,conventional hydroelectric


In [ ]:
#fueltypeid = fuel_category
#fuelTypeDescription = fuel_name

In [7]:
fuel_df["fuelShort"] =fuel_df["product-name"].map({"Natural Gas": "NG"})
fuel_df.head()

,period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units,fuelShort
0,2025-03,SAR,USA-AR,EPG0,Natural Gas,PEU,Electric Power Price,N3045AR3,Arkansas Natural Gas Price Sold to Electric Po...,NaN,$/MCF,NG
1,2025-03,SNE,USA-NE,EPG0,Natural Gas,PEU,Electric Power Price,N3045NE3,Nebraska Natural Gas Price Sold to Electric Po...,7.28,$/MCF,NG
2,2025-03,SMI,USA-MI,EPG0,Natural Gas,PEU,Electric Power Price,N3045MI3,Michigan Natural Gas Price Sold to Electric Po...,NaN,$/MCF,NG
3,2025-03,SCO,COLORADO,EPG0,Natural Gas,PEU,Electric Power Price,N3045CO3,Colorado Natural Gas Price Sold to Electric Po...,NaN,$/MCF,NG
4,2025-03,SAK,USA-AK,EPG0,Natural Gas,PEU,Electric Power Price,N3045AK3,Alaska Natural Gas Price Sold to Electric Powe...,8.34,$/MCF,NG


### Clean The Date

In [12]:
gen_df["dateShort"] = gen_df.period.str.replace("-", "")
gen_df.head()

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units,dateShort
0,2025-03,90,Pacific,1,Electric Utility,ALL,all fuels,17842.73950,thousand megawatthours,202503
1,2025-03,90,Pacific,1,Electric Utility,AOR,all renewables,822.00528,thousand megawatthours,202503
2,2025-03,90,Pacific,1,Electric Utility,BIO,biomass,37.41079,thousand megawatthours,202503
3,2025-03,90,Pacific,1,Electric Utility,COL,"coal, excluding waste coal",36.42772,thousand megawatthours,202503
4,2025-03,90,Pacific,1,Electric Utility,COW,all coal products,36.42772,thousand megawatthours,202503


### Sectors
-generation sector and price sector are not the same

In [14]:
#Generation Sectors
gen_df.sectorDescription.unique()

<StringArray>
[             'Electric Utility',                   'IPP Non-CHP',
                       'IPP CHP',            'Commercial Non-CHP',
                'Commercial CHP',            'Industrial Non-CHP',
                'Industrial CHP', 'Electric Power Sector Non-CHP',
   'Independent Power Producers',              'Coal Consumption',
                'All Commercial',                'All Industrial',
                'Electric Power',                   'All Sectors',
                   'Residential']
Length: 15, dtype: str

In [15]:
price_df.sectorName.unique()

<StringArray>
[   'all sectors',     'commercial',     'industrial',          'other',
    'residential', 'transportation']
Length: 6, dtype: str

In [22]:
idmap = {
    "statemap": {"KS": "Kansas", "MO": "Missouri"},
    "fuelmap": {"NG": "Natural Gas", "CO":"Coal"}
}

In [23]:
idmap["statemap"]

{'KS': 'Kansas', 'MO': 'Missouri'}

### Dealing with Fuel ID


In [24]:
gen_df.head()

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units,dateShort
0,2025-03,90,Pacific,1,Electric Utility,ALL,all fuels,17842.73950,thousand megawatthours,202503
1,2025-03,90,Pacific,1,Electric Utility,AOR,all renewables,822.00528,thousand megawatthours,202503
2,2025-03,90,Pacific,1,Electric Utility,BIO,biomass,37.41079,thousand megawatthours,202503
3,2025-03,90,Pacific,1,Electric Utility,COL,"coal, excluding waste coal",36.42772,thousand megawatthours,202503
4,2025-03,90,Pacific,1,Electric Utility,COW,all coal products,36.42772,thousand megawatthours,202503


In [25]:
gen_df.fueltypeid.unique()

<StringArray>
['ALL', 'AOR', 'BIO', 'COL', 'COW', 'DFO', 'FOS', 'GEO', 'HPS', 'HYC', 'LFG',
 'LIG', 'MLG',  'NG', 'NGO', 'NUC', 'OB2', 'OBW', 'OOG', 'ORW', 'OTH', 'PEL',
 'PET', 'REN', 'RFO', 'SPV', 'SUB', 'SUN', 'WAS', 'WND', 'WNT', 'WOC', 'WOO',
 'WWW', 'MSB',  'PC',  'RC', 'STH', 'BIS', 'BIT', 'DPV', 'TPV', 'TSN', 'ANT',
 'WNS']
Length: 45, dtype: str